# RAG over a PDF with Azure AI Content Understanding + Azure AI Search

**Flow:** PDF → Content Understanding (`prebuilt-layout`) → Markdown → header-based chunks → embeddings → Azure AI Search index → retrieval → grounded answer.

Modernized version of `search_with_document_layout.ipynb` from the cloned sample:

| Original sample | This notebook |
| --- | --- |
| Custom REST client, preview API `2024-12-01-preview`, custom analyzer created/deleted from a JSON template | `azure-ai-contentunderstanding` 1.1.0 (GA API `2025-11-01`), built-in `prebuilt-layout` analyzer, nothing to create or delete |
| LangChain `AzureSearch`, `AzureChatOpenAI`, `hub.pull`, legacy `langchain.*` imports | `azure-search-documents` 12 (explicit index schema), `openai` SDK, `langchain-text-splitters` for chunking only |
| `AzureOpenAI*` classes pinned to dated api-versions | `OpenAI` client on the `/openai/v1/` endpoint (no api-version) |
| `%pip install` inside the notebook | Dependencies live in `pyproject.toml` (managed by `uv`) |

## Prerequisites

1. **Azure resources (portal)**
   - Microsoft Foundry resource in a region that supports Content Understanding. Copy its endpoint (`https://<name>.services.ai.azure.com/`).
   - In that resource, deploy an embedding model (e.g. `text-embedding-3-large`) and a chat model (e.g. `gpt-4.1-mini`).
   - Azure AI Search service. Under **Settings → Keys → API access control**, choose *Role-based access control* (or *Both*).
2. **Roles for your user**
   - Foundry resource: `Cognitive Services User`
   - Search service: `Search Service Contributor` and `Search Index Data Contributor`
3. **Sign in:** run `az login` in a terminal.
4. **Config:** copy `.env.sample` to `.env` (project root) and fill in the values.
5. **Kernel:** select the `adi` kernel. No `pip install` is needed.

## 1. Configuration

In [1]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing environment variable '{name}'. See .env.sample.")
    return value


CU_ENDPOINT = require_env("CONTENTUNDERSTANDING_ENDPOINT")
AOAI_ENDPOINT = require_env("AZURE_OPENAI_ENDPOINT")
EMBEDDING_DEPLOYMENT = require_env("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
CHAT_DEPLOYMENT = require_env("AZURE_OPENAI_CHAT_DEPLOYMENT")
SEARCH_ENDPOINT = require_env("AZURE_SEARCH_ENDPOINT")
INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "sample-doc-index")

DOC_PATH = Path("../data/sample_layout.pdf")
assert DOC_PATH.exists(), f"File not found: {DOC_PATH.resolve()}"

# Must match the index vector field. 3072 is the default for text-embedding-3-large.
EMBEDDING_DIMENSIONS = 3072

## 2. Create clients

`DefaultAzureCredential` picks up your `az login` session and backs all three services. The OpenAI client receives a bearer-token provider instead of an API key.

In [2]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from openai import OpenAI

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

cu_client = ContentUnderstandingClient(endpoint=CU_ENDPOINT, credential=credential)

openai_client = OpenAI(
    base_url=f"{AOAI_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential)

## 3. Extract Markdown with Content Understanding

`prebuilt-layout` runs OCR and layout analysis (paragraphs, tables, sections) and returns Markdown. It needs **no model deployments**.

Analysis is a long-running operation; `poller.result()` blocks until it finishes. A PDF yields one content item that spans all pages.

In [3]:
from azure.ai.contentunderstanding.models import AnalysisResult, DocumentContent

ANALYZER_ID = "prebuilt-layout"  # or "prebuilt-documentSearch" (see 3a)

poller = cu_client.begin_analyze_binary(
    analyzer_id=ANALYZER_ID,
    binary_input=DOC_PATH.read_bytes(),
)
result: AnalysisResult = poller.result()

doc = result.contents[0]
assert isinstance(doc, DocumentContent)
markdown_content = doc.markdown

print(f"Pages {doc.start_page_number}-{doc.end_page_number} | tables: {len(doc.tables or [])} | markdown chars: {len(markdown_content)}")
print(markdown_content[:2000])

Pages 1-1 | tables: 1 | markdown chars: 1571
<!-- PageHeader: This is the header of the document. -->


# This is title


## 1. Text

Latin refers to an ancient Italic language
originating in the region of Latium in
ancient Rome.


## 2. Page Objects


### 2.1 Table

Here's a sample table below, designed to
be simple for easy understand and quick
reference.


<table>
<caption>Table 1: This is a dummy table</caption>
<tr>
<td>Name</td>
<td>Corp</td>
<td>Remark</td>
</tr>
<tr>
<td>Foo</td>
<td></td>
<td></td>
</tr>
<tr>
<td>Bar</td>
<td>Microsoft</td>
<td>Dummy</td>
</tr>
</table>


### 2.2. Figure


Figure 1: Here is a figure with text

![Values 500 450 400 400 350 300 300 250 200 200 100 0 Jan Feb Mar Apr May Jun Months](figures/1.1)


## 3. Others

Al Document Intelligence is an Al service
that applies advanced machine learning
to extract text, key-value pairs, tables,
and structures from documents
automatically and accurately:

☑
clear

☑
precise

☐
vague

☑
coherent

☐
Incomprehensi

In [9]:
type(result)

azure.ai.contentunderstanding.models._models.AnalysisResult

### 3a. Optional: try `prebuilt-documentSearch`

`prebuilt-documentSearch` is the RAG-oriented analyzer: the same Markdown plus figure/chart descriptions and a summary. It calls your model deployments, so map them once per Foundry resource (needs the `Cognitive Services User` role). Then set `ANALYZER_ID = "prebuilt-documentSearch"` in step 3 and re-run from there.

In [11]:
# One-time per Foundry resource. Not needed for prebuilt-layout.
COMPLETION_MODEL = "gpt-5-mini"  # model name you deployed, not the deployment name
EMBEDDING_MODEL = "text-embedding-3-large"

defaults = cu_client.update_defaults(
    model_deployments={
        COMPLETION_MODEL: CHAT_DEPLOYMENT,
        EMBEDDING_MODEL: EMBEDDING_DEPLOYMENT,
        "prebuilt-analyzer-completion": CHAT_DEPLOYMENT,
        "prebuilt-analyzer-completion-mini": CHAT_DEPLOYMENT,
        "prebuilt-analyzer-embedding": EMBEDDING_DEPLOYMENT,
    }
)
print(defaults.model_deployments)

{'gpt-5.4': 'gpt-5.4', 'gpt-5-mini': 'gpt-5-mini', 'text-embedding-3-large': 'text-embedding-3-large', 'prebuilt-analyzer-completion': 'gpt-5-mini', 'prebuilt-analyzer-completion-mini': 'gpt-5-mini', 'prebuilt-analyzer-embedding': 'text-embedding-3-large', 'gpt-4.1-mini': 'gpt-5-mini'}


## 4. Split into chunks

Splitting on Markdown headers keeps each section together and stores the heading path as metadata. A second pass caps the size of long sections (very large tables can be split by this cap too).

In [18]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

HEADERS_TO_SPLIT_ON = [("#", "header_1"), ("##", "header_2"), ("###", "header_3")]
CHUNK_SIZE = 100  # characters (roughly 500 tokens)
CHUNK_OVERLAP = 30

header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS_TO_SPLIT_ON, strip_headers=False)
size_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

sections = header_splitter.split_text(markdown_content)
chunks = size_splitter.split_documents(sections)

print(f"{len(sections)} sections -> {len(chunks)} chunks")
for chunk in chunks[:8]:
    print(chunk.metadata, "|", chunk.page_content[:120].replace("\n", " "), "\n")

5 sections -> 22 chunks
{} | <!-- PageHeader: This is the header of the document. --> 

{'header_1': 'This is title', 'header_2': '1. Text'} | # This is title   ## 1. Text   Latin refers to an ancient Italic language 

{'header_1': 'This is title', 'header_2': '1. Text'} | originating in the region of Latium in ancient Rome. 

{'header_1': 'This is title', 'header_2': '2. Page Objects', 'header_3': '2.1 Table'} | ## 2. Page Objects   ### 2.1 Table   Here's a sample table below, designed to 

{'header_1': 'This is title', 'header_2': '2. Page Objects', 'header_3': '2.1 Table'} | be simple for easy understand and quick reference.   <table> 

{'header_1': 'This is title', 'header_2': '2. Page Objects', 'header_3': '2.1 Table'} | reference.   <table> <caption>Table 1: This is a dummy table</caption> <tr> <td>Name</td> 

{'header_1': 'This is title', 'header_2': '2. Page Objects', 'header_3': '2.1 Table'} | <tr> <td>Name</td> <td>Corp</td> <td>Remark</td> </tr> <tr> <td>Foo</td> <td></td> <

## 5. Create the Azure AI Search index

The schema has a keyword-searchable `content` field, heading metadata for filtering, an HNSW vector field, and a semantic configuration. `create_or_update_index` is safe to re-run; changing the vector dimensions of an existing index requires deleting it first.

In [ ]:
from azure.search.documents.indexes.models import (
    HnswAlgorithmConfiguration,
    SearchField,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    VectorSearch,
    VectorSearchProfile,
)

fields = [
    SearchField(name="id", type="Edm.String", key=True, filterable=True),
    SearchField(name="content", type="Edm.String", searchable=True),
    SearchField(name="header_1", type="Edm.String", searchable=True, filterable=True),
    SearchField(name="header_2", type="Edm.String", searchable=True, filterable=True),
    SearchField(name="header_3", type="Edm.String", searchable=True, filterable=True),
    SearchField(name="source", type="Edm.String", filterable=True),
    SearchField(
        name="content_vector",
        type="Collection(Edm.Single)",
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="hnsw-profile",
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
    profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw-config")],
)

semantic_search = SemanticSearch(
    default_configuration_name="semantic-config",
    configurations=[
        SemanticConfiguration(
            name="semantic-config",
            prioritized_fields=SemanticPrioritizedFields(
                title_field=SemanticField(field_name="header_1"),
                content_fields=[SemanticField(field_name="content")],
            ),
        )
    ],
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)
index_client.create_or_update_index(index)
print(f"Index '{INDEX_NAME}' is ready")

## 6. Embed the chunks and upload them

Chunk IDs are deterministic (`<file>-<n>`), so re-running this cell updates existing documents instead of duplicating them. Newly uploaded documents can take a few seconds to become searchable.

In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=texts,
        dimensions=EMBEDDING_DIMENSIONS,  # supported by text-embedding-3-* only
    )
    return [item.embedding for item in response.data]


BATCH_SIZE = 16
documents = []
for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start : start + BATCH_SIZE]
    vectors = embed([chunk.page_content for chunk in batch])
    for offset, (chunk, vector) in enumerate(zip(batch, vectors)):
        documents.append(
            {
                "id": f"{DOC_PATH.stem}-{start + offset:04d}",
                "content": chunk.page_content,
                "header_1": chunk.metadata.get("header_1"),
                "header_2": chunk.metadata.get("header_2"),
                "header_3": chunk.metadata.get("header_3"),
                "source": DOC_PATH.name,
                "content_vector": vector,
            }
        )

upload_results = search_client.merge_or_upload_documents(documents=documents)
print(f"Uploaded {sum(r.succeeded for r in upload_results)}/{len(documents)} chunks to '{INDEX_NAME}'")

## 7. Retrieve relevant chunks

Three query modes against the same index:

| Mode | What it does |
| --- | --- |
| `vector` | Nearest-neighbour search on `content_vector` only |
| `hybrid` | Keyword (BM25) + vector, fused with Reciprocal Rank Fusion |
| `semantic` | Hybrid, then re-ranked by the semantic ranker (must be enabled on the search service) |

In [ ]:
from azure.search.documents.models import VectorizedQuery

SELECT_FIELDS = ["id", "content", "header_1", "header_2", "header_3", "source"]


def retrieve(query: str, k: int = 3, mode: str = "semantic") -> list[dict]:
    """mode: 'vector', 'hybrid' (keyword + vector) or 'semantic' (hybrid + semantic reranker)."""
    vector_query = VectorizedQuery(
        vector=embed([query])[0],
        k_nearest_neighbors=k,
        fields="content_vector",
    )
    params = {
        "search_text": None if mode == "vector" else query,
        "vector_queries": [vector_query],
        "select": SELECT_FIELDS,
        "top": k,
    }
    if mode == "semantic":
        params.update(query_type="semantic", semantic_configuration_name="semantic-config")
    return list(search_client.search(**params))

In [ ]:
QUESTION = "Summarize the key points of this document."  # replace with a question about your PDF

for mode in ("vector", "hybrid", "semantic"):
    print(f"--- {mode} ---")
    for hit in retrieve(QUESTION, mode=mode):
        score = hit.get("@search.reranker_score") or hit["@search.score"]
        print(f"{score:.3f} | {hit['id']} | {hit['header_1']} > {hit['header_2']}")

## 8. Generate a grounded answer

The retrieved chunks are passed to the chat model as context. `answer()` returns both the text and the source chunks, which step 10 uses for references.

In [ ]:
SYSTEM_PROMPT = (
    "You answer questions about a document using only the provided context. "
    "If the context does not contain the answer, say you don't know. Keep answers concise."
)


def answer(question: str, k: int = 3, mode: str = "semantic") -> dict:
    hits = retrieve(question, k=k, mode=mode)
    context = "\n\n".join(hit["content"] for hit in hits)
    # No temperature set: GPT-5 family models only accept the default.
    response = openai_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return {"answer": response.choices[0].message.content, "sources": hits}

## 9. Document Q&A

In [ ]:
print(answer(QUESTION)["answer"])

## 10. Q&A with references

In [ ]:
response = answer(QUESTION)
print(response["answer"])
print("\nSources:")
for hit in response["sources"]:
    print(f"- {hit['id']} ({hit['source']}): {hit['header_1']} > {hit['header_2']} > {hit['header_3']}")

## Clean up (optional)

In [ ]:
# Uncomment to delete the index. The search service keeps billing until the resource itself is deleted.
# index_client.delete_index(INDEX_NAME)

## Ideas to explore next

- Switch `ANALYZER_ID` to `prebuilt-documentSearch` (step 3a) and compare the Markdown: figure descriptions, summary.
- Compare `vector`, `hybrid` and `semantic` retrieval on the same questions; vary `k`, `CHUNK_SIZE` and `CHUNK_OVERLAP`.
- Add a metadata filter to `retrieve()`, e.g. `filter="header_1 eq '...'"`.
- Try the beta SDK (`azure-ai-contentunderstanding==1.2.0b3`, API `2026-06-01-preview`): built-in `SemanticChunkingStrategy` and `to_llm_input()` can replace the manual chunking step.
- Let Azure AI Search ingest for you: the Content Understanding skill in an indexer (blob storage → skillset → index).